# Lesson 4.3 — Data Binding: Metrics Queries

## Learning Objectives
- Configure Server Metrics data binding in Dash Studio widgets
- Configure Custom Query (BT metric) data binding
- Understand the difference between Data Queries, Custom Queries, and ADQL Queries
- Use the Metric Browser to find the exact metric path for a widget

**Reference**: [Data Binding](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/dashboards-and-reports/dash-studio/data-binding) | [Data Queries](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/dashboards-and-reports/dash-studio/data-binding/data-queries)

## 1. Data Binding in Dash Studio

**Data binding** connects a widget to its data source. Every widget in Dash Studio needs a data binding to show anything.

### The Data Panel
When you click a widget in Edit mode, the right panel shows:
1. **Show me data for** — the data source dropdown
2. Data-source-specific fields (metric path, application, BT name, etc.)
3. **Display options** — title, colors, thresholds

### The Five Main Data Sources

| Data Source | What it queries | When to use |
|-------------|-----------------|-------------|
| **Data Queries** | Predefined metric categories | Quick setup for common metrics |
| **Server Metrics** | Machine Agent host metrics | CPU, memory, disk per server |
| **Custom Query** | Any metric path with application+tier+BT picker | BT-specific metrics |
| **ADQL Query** | Full ADQL on Events Service | Analytics data, time series (Lesson 4.4) |
| **Database Metrics** | Database Visibility data | Query performance, wait states |

## 2. Custom Query Data Binding

The **Custom Query** data source lets you select a specific metric by navigating:
1. Application → Tier → Business Transaction → Metric

### Step-by-step for a BT response time widget:

1. Add a Time Series widget to the canvas
2. In the Data panel → **Show me data for: Custom Query**
3. **Application**: Select `StoreFront`
4. **Metric path** dropdown:
   - Navigate: `Business Transaction Performance` → `Business Transactions` → `WebFrontend` → `/checkout`
   - Select: `Average Response Time (ms)`
5. **Aggregation**: Sum, Average, Min, Max, Count (default: Current)
6. **Label**: Name this series (e.g., "/checkout Avg RT")
7. Click **+ Add another metric** to add `/search` on the same chart

### Resulting metric path (displayed in Data panel):
```
Business Transaction Performance|Business Transactions|WebFrontend|/checkout|Average Response Time (ms)
```

In [ ]:
# Reference: Build valid metric paths programmatically
# Useful for verifying paths before pasting into Dash Studio

def build_metric_paths(application_name: str, tier: str, bt_name: str) -> dict:
    """
    Build the standard AppDynamics metric paths for a Business Transaction.
    Returns a dict of metric_name → full_path.
    """
    base = f"Business Transaction Performance|Business Transactions|{tier}|{bt_name}"
    return {
        "avg_response_time":   f"{base}|Average Response Time (ms)",
        "calls_per_minute":    f"{base}|Calls per Minute",
        "errors_per_minute":   f"{base}|Errors per Minute",
        "slow_calls":          f"{base}|Number of Slow Calls",
        "very_slow_calls":     f"{base}|Number of Very Slow Calls",
        "stall_count":         f"{base}|Stall Count",
    }

# Example: metrics for /checkout in WebFrontend tier of StoreFront
checkout_metrics = build_metric_paths("StoreFront", "WebFrontend", "/checkout")

print("Metric paths for StoreFront > WebFrontend > /checkout:")
print()
for name, path in checkout_metrics.items():
    print(f"  {name}:")
    print(f"    {path}")
    print()

In [ ]:
# Reference: Server Metrics data binding paths
# For the Machine Agent infrastructure metrics

def build_server_metric_paths(app_name: str, tier: str, node: str) -> dict:
    """Build common Server/Infrastructure metric paths."""
    base = f"Application Infrastructure Performance|{tier}|Individual Nodes|{node}"
    return {
        "cpu_busy_pct":        f"{base}|Hardware Resources|CPU|%Busy",
        "memory_used_mb":      f"{base}|Hardware Resources|Memory|Used (MB)",
        "jvm_heap_mb":         f"{base}|JVM|Heap|Current Usage (MB)",
        "jvm_gc_time":         f"{base}|JVM|Garbage Collection|GC Time Spent Per Minute (ms)",
        "thread_count":        f"{base}|JVM|Threads|Current No. of Threads",
    }

node_metrics = build_server_metric_paths("StoreFront", "OrderService", "order-svc-1")

print("Server metric paths for StoreFront > OrderService > order-svc-1:")
for name, path in node_metrics.items():
    print(f"  {name}:")
    print(f"    {path}")

## 3. Server Metrics Data Binding

### Step-by-step for a CPU chart per node:

1. Add a Time Series widget
2. Data panel → **Show me data for: Server Metrics**
3. **Application**: `StoreFront`
4. **Tier**: `OrderService`
5. **Node**: `order-svc-1` (or use `*` for all nodes in the tier)
6. **Metric**: `Hardware Resources | CPU | %Busy`
7. Optionally click **+ Add another metric** and add `Memory | Used (MB)` on a second Y-axis

> **Tip**: Use `*` (wildcard) for the Node field to show ALL nodes in a tier on the same chart. This helps spot if one node is behaving differently from the others.

In [ ]:
# REST API: Verify a metric path exists before using it in a dashboard

import os, requests

CONTROLLER_URL = os.getenv("APPD_CONTROLLER_URL", "https://your-account.saas.appdynamics.com")
ACCOUNT_NAME   = os.getenv("APPD_ACCOUNT", "your-account")
USERNAME       = os.getenv("APPD_USER", "admin")
PASSWORD       = os.getenv("APPD_PASSWORD", "")
APP_NAME       = os.getenv("APPD_APP_NAME", "StoreFront")

def verify_metric_path(app_name: str, metric_path: str) -> bool:
    """
    Verify that a metric path exists and has data in the last 15 minutes.
    Returns True if data exists, False otherwise.
    """
    url = f"{CONTROLLER_URL}/controller/rest/applications/{app_name}/metric-data"
    params = {
        "metric-path": metric_path,
        "time-range-type": "BEFORE_NOW",
        "duration-in-mins": 15,
        "output": "JSON"
    }
    auth = (f"{USERNAME}@{ACCOUNT_NAME}", PASSWORD)
    try:
        resp = requests.get(url, auth=auth, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        return bool(data and data[0].get('metricValues'))
    except:
        return False

# Test a set of metric paths
test_paths = [
    "Overall Application Performance|Average Response Time (ms)",
    "Business Transaction Performance|Business Transactions|WebFrontend|/checkout|Average Response Time (ms)",
    "Business Transaction Performance|Business Transactions|WebFrontend|/NONEXISTENT|Average Response Time (ms)",
]

print("Metric path validation:")
for path in test_paths:
    # Simulate result since we're not connected
    has_data = "NONEXISTENT" not in path  # In real code: verify_metric_path(APP_NAME, path)
    status = "✅ Valid" if has_data else "❌ No data / Invalid"
    print(f"  {status}: {path[:70]}{'...' if len(path) > 70 else ''}")

In [ ]:
# EXERCISE 4.3: Configure the data binding for 3 widgets
#
# For each widget below, provide:
#   1. The correct data source type (Custom Query / Server Metrics / ADQL Query)
#   2. The full metric path OR ADQL query

widgets_to_configure = [
    {
        "widget": "Time Series: /checkout avg response time trend",
        "data_source": "???",   # TODO
        "query_or_path": "???", # TODO
    },
    {
        "widget": "Stat: CPU % busy for OrderService tier (average across all nodes)",
        "data_source": "???",   # TODO: hint: Server Metrics with wildcard node
        "query_or_path": "???", # TODO: Include the metric path
    },
    {
        "widget": "Table: Top 5 BTs by total error count in the last hour",
        "data_source": "???",   # TODO: This needs Events Service data
        "query_or_path": "???", # TODO: Write the ADQL
    },
]

print("Data binding configuration for 3 widgets:")
print("=" * 60)
for w in widgets_to_configure:
    print(f"\nWidget: {w['widget']}")
    print(f"  Data source: {w['data_source']}")
    print(f"  Path/Query:  {w['query_or_path']}")

## Summary

- Data binding connects widgets to AppDynamics data via **Custom Query**, **Server Metrics**, **ADQL Query**, or **Data Queries**
- **Custom Query**: Navigate Application → Tier → BT → metric — best for BT and application-level metrics
- **Server Metrics**: Machine Agent host metrics (CPU, memory, JVM)
- Use the `*` wildcard for Node to show all nodes in a tier on one chart
- Always verify metric paths in the **Metric Browser** before adding them to a dashboard

---

**Next**: [Lesson 4.4 — ADQL Queries in Dash Studio](lesson-4.4-adql-in-dash-studio.ipynb)